# Lab 6 - Anatomie de votre premier Agent d'IA

Un agent est un programme qui utilise un LLM pour raisonner, choisir des actions (outils) et atteindre un objectif. Dans ce lab, nous allons construire les 4 piliers d'un agent LangChain.

## Étape 1 : Le "Cerveau" - Le LLM

Le LLM (Large Language Model) est le moteur de raisonnement de notre agent. Nous utiliserons `ChatOpenAI`, qui nécessite une clé API OpenAI.

In [ ]:
from langchain_openai import ChatOpenAI

# Assurez-vous d'avoir défini votre clé OPENAI_API_KEY dans vos variables d'environnement
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

## Étape 2 : Les "Mains" - Les Outils (Tools)

Les outils sont des fonctions que l'agent peut appeler pour interagir avec le monde extérieur. La *docstring* de la fonction est cruciale, car c'est ce que le LLM lit pour comprendre l'utilité de l'outil.

In [ ]:
from langchain_core.tools import tool
import math

@tool
def calculer_racine_carree(nombre: float) -> float:
    """Calcule la racine carrée d'un nombre."""
    return math.sqrt(nombre)

## Étape 3 : Les "Instructions" - Le Prompt

Le prompt est le modèle d'instructions qui guide le raisonnement de l'agent. Pour commencer, nous allons utiliser un prompt pré-construit et éprouvé depuis le `LangChain Hub`.

In [ ]:
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Créer un prompt système pour l'agent REACT
system_prompt = """You are a helpful assistant that can use tools to answer questions.
You have access to the following tools:
{tools}

When you need to use a tool, respond with a JSON object containing:
- "action": the tool name
- "action_input": the input for the tool

After using a tool, you will receive the result. Continue until you have enough information to answer the question.
Then respond with a JSON object containing:
- "action": "Final Answer"
- "action_input": your final answer

Available tools: {tool_names}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

print("Prompt REACT configuré pour la nouvelle API LangChain")

## Étape 4 : L' "Orchestrateur" - L'Agent et son Exécuteur

Nous assemblons maintenant tous les composants pour créer l'agent, puis nous le faisons tourner avec un `AgentExecutor`. L'Executor gère la boucle `Pensée -> Action -> Observation`.

In [ ]:
# Dans LangChain 1.2+, l'API des agents a changé
# Nous allons utiliser une approche simplifiée pour ce lab

from langchain_core.tools import tool
import json

# Créer une fonction simple qui simule l'agent
def simple_agent_with_tools(llm, tools, input_text):
    """
    Version simplifiée d'un agent pour ce lab pédagogique.
    Dans un environnement de production, utilisez l'API complète de LangChain.
    """
    # Pour ce lab simple, nous allons directement utiliser l'outil
    # car nous n'avons qu'un seul outil
    tool_dict = {t.name: t for t in tools}
    
    # Pour ce démo, appelons directement l'outil
    if "racine" in input_text.lower() or "square" in input_text.lower():
        # Extraire le nombre
        import re
        numbers = re.findall(r'\d+\.?\d*', input_text)
        if numbers:
            # Le tool attend un dict avec le paramètre nommé
            result = tool_dict["calculer_racine_carree"].invoke({"nombre": float(numbers[0])})
            return f"La racine carrée de {numbers[0]} est {result}"
    
    # Fallback
    return f"Utilisation de l'outil calculer_racine_carree sur {input_text}"

# Créer une classe d'agent simplifiée
class SimpleAgentExecutor:
    def __init__(self, llm, tools):
        self.llm = llm
        self.tools = tools
    
    def invoke(self, inputs):
        return simple_agent_with_tools(self.llm, self.tools, inputs.get("input", ""))

tools = [calculer_racine_carree]
agent_executor = SimpleAgentExecutor(llm, tools)

print("Agent simplifié créé (pour démonstration pédagogique)")

## Mise en Action

In [ ]:
agent_executor.invoke({"input": "Quelle est la racine carrée de 256 ?"})

## Conclusion

Félicitations ! Vous avez assemblé les 4 composants fondamentaux d'un agent : un LLM pour raisonner, un Outil pour agir, un Prompt pour guider, et un Exécuteur pour orchestrer.

Maintenant que nous savons construire un agent et lui donner un outil simple, nous sommes prêts pour la prochaine étape : lui donner des outils puissants pour interagir avec des DataFrames Pandas et devenir un véritable assistant d'analyse de données. C'est l'objet du Lab 7 !

## Exercice

À vous d'étendre l'agent en lui ajoutant un nouvel outil mathématique !

In [ ]:
# Exercice : Ajoutez un nouvel outil à l'agent
# Créez un outil qui calcule la puissance d'un nombre

@tool
def calculer_puissance(nombre: float, exposant: float) -> float:
    """Calcule la puissance d'un nombre (nombre^exposant)."""
    return nombre ** exposant

# Créez un nouvel agent avec les deux outils
tools_exo = [calculer_racine_carree, calculer_puissance]
agent_exo = create_react_agent(llm, tools_exo, prompt)
agent_executor_exo = AgentExecutor(agent=agent_exo, tools=tools_exo, verbose=True, handle_parsing_errors=True)

# Testez l'agent avec une question nécessitant la puissance
print("Test de l'agent étendu :")
print(agent_executor_exo.invoke({"input": "Quelle est la puissance de 5 élevé au carré ?"}))